# The Quiet Epidemic
## How Deindustrialization Killed Working-Class America — One Community at a Time

*Data journalism study | Deaths of Despair 1999–2016 | Sources: CDC NCHS, BLS, Census Bureau*

In [1]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"

import sys
sys.path.insert(0, "/Users/matt/data-adventures")

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from pipeline.config import load_config, get_data_processed_dir

PROJECT = "deaths-of-despair"
config = load_config(PROJECT)
processed_dir = get_data_processed_dir(config)

state_panel = pd.read_parquet(processed_dir / "state_panel.parquet")
national_panel = pd.read_parquet(processed_dir / "national_panel.parquet")

# Clean state panel to key indicators
state_panel = state_panel.copy()
state_panel["year"] = state_panel["year"].astype(int)
national_panel = national_panel.copy()
national_panel["year"] = national_panel["year"].astype(int)

# Build 2000-baseline change metrics for animation
base_2000 = (
    state_panel[state_panel["year"] == 2000]
    [["state_abbr","deaths_despair_rate","manufacturing_employees_thousands"]]
    .rename(columns={
        "deaths_despair_rate": "ddr_2000",
        "manufacturing_employees_thousands": "mfg_2000",
    })
)

panel = state_panel.merge(base_2000, on="state_abbr", how="left")
panel["ddr_change_since_2000"] = panel["deaths_despair_rate"] - panel["ddr_2000"]
panel["mfg_change_pct_2000"] = (
    (panel["manufacturing_employees_thousands"] - panel["mfg_2000"]) / panel["mfg_2000"] * 100
)

# Summary stats
total_years = int(state_panel.year.max()) - int(state_panel.year.min())
nat = national_panel.sort_values("year")
start_rate = nat["deaths_despair_rate"].iloc[0]
end_rate = nat["deaths_despair_rate"].iloc[-1]
total_deaths_sum = national_panel["overdose_deaths_total"].sum() + national_panel["suicide_deaths_total"].sum()

print(f"Study period: {int(state_panel.year.min())}–{int(state_panel.year.max())} ({total_years} years)")
print(f"National deaths of despair rate: {start_rate:.1f} → {end_rate:.1f} per 100k ({(end_rate/start_rate - 1)*100:.0f}% increase)")
print(f"Estimated total overdose + suicide deaths in dataset: {total_deaths_sum:,.0f}")


Study period: 1999–2016 (17 years)
National deaths of despair rate: 17.6 → 36.9 per 100k (110% increase)
Estimated total overdose + suicide deaths in dataset: 1,282,174


## Chapter 1: The Scale of the Crisis

From 1999 to 2016, the combined rate of drug overdoses and suicides nearly doubled across the United States. This isn't a single incident, a single city, or a single drug. It's a systematic collapse of communities left behind.

In [2]:
nat_sorted = national_panel.dropna(subset=["deaths_despair_rate"]).sort_values("year")

fig = go.Figure()

# Background shading for waves
fig.add_vrect(x0=1999, x1=2010, fillcolor="#fde8d8", opacity=0.3, line_width=0,
    annotation_text="Wave 1:<br>Rx Opioids", annotation_position="top left",
    annotation=dict(font_size=11, font_color="#e67e22"))
fig.add_vrect(x0=2010, x1=2013, fillcolor="#fad7d7", opacity=0.3, line_width=0,
    annotation_text="Wave 2:<br>Heroin", annotation_position="top left",
    annotation=dict(font_size=11, font_color="#c0392b"))
fig.add_vrect(x0=2013, x1=2016, fillcolor="#e8d0f5", opacity=0.3, line_width=0,
    annotation_text="Wave 3:<br>Fentanyl", annotation_position="top left",
    annotation=dict(font_size=11, font_color="#8e44ad"))

fig.add_trace(go.Scatter(
    x=nat_sorted["year"], y=nat_sorted["overdose_death_rate"],
    mode="lines+markers", name="Drug Overdose Deaths",
    line=dict(color="#e74c3c", width=3),
    fill="tozeroy", fillcolor="rgba(231,76,60,0.08)",
))
fig.add_trace(go.Scatter(
    x=nat_sorted["year"], y=nat_sorted["suicide_rate"],
    mode="lines+markers", name="Suicide Deaths",
    line=dict(color="#8e44ad", width=3),
    fill="tozeroy", fillcolor="rgba(142,68,173,0.08)",
))
fig.add_trace(go.Scatter(
    x=nat_sorted["year"], y=nat_sorted["deaths_despair_rate"],
    mode="lines", name="Combined: Deaths of Despair",
    line=dict(color="#2c3e50", width=4, dash="dot"),
))

fig.update_layout(
    title=dict(text="The Quiet Epidemic — Deaths of Despair, United States 1999–2016", font_size=18),
    xaxis=dict(title="Year", tickmode="linear", dtick=2),
    yaxis=dict(title="Age-adjusted deaths per 100,000 people"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
    height=500,
    hovermode="x unified",
    plot_bgcolor="#fafafa",
)
fig.show()


## Chapter 2: Watch It Spread — An American Catastrophe in Motion

This animated map shows how deaths of despair spread across the United States, year by year. The darkening red is not random — it follows the geography of economic collapse.

In [3]:
# Animated choropleth — deaths of despair by state over time
anim_data = state_panel.dropna(subset=["deaths_despair_rate","state_abbr"]).sort_values(["year","state_abbr"])
max_rate = anim_data["deaths_despair_rate"].quantile(0.97)

fig = px.choropleth(
    anim_data,
    locations="state_abbr",
    locationmode="USA-states",
    color="deaths_despair_rate",
    animation_frame="year",
    scope="usa",
    color_continuous_scale=[
        [0.0, "#fff5f0"], [0.15, "#fee0d2"], [0.3, "#fcbba1"],
        [0.45, "#fc9272"], [0.6, "#fb6a4a"], [0.75, "#ef3b2c"],
        [0.9, "#cb181d"], [1.0, "#67000d"],
    ],
    range_color=[0, max_rate],
    hover_data={
        "state_abbr": False,
        "state_name": True,
        "overdose_death_rate": ":.1f",
        "suicide_rate": ":.1f",
        "deaths_despair_rate": ":.1f",
    },
    labels={
        "deaths_despair_rate": "Deaths per 100k",
        "overdose_death_rate": "Overdose rate",
        "suicide_rate": "Suicide rate",
        "state_name": "State",
    },
    title="Deaths of Despair per 100,000 Americans — Animated 1999 to 2016",
)
fig.update_layout(
    height=580,
    coloraxis_colorbar=dict(title="Deaths per 100k", tickfont_size=11),
    geo=dict(bgcolor="#f0f4f8"),
)
# Slow down animation
fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 600
fig.layout.updatemenus[0].buttons[0].args[1]["transition"]["duration"] = 300
fig.show()


## Chapter 3: The Economic Blueprint

The map above doesn't look like a random health crisis. It looks like a map of where factories used to be.

Watch below as states that lost the most manufacturing jobs ended up with the highest deaths of despair. This animation tracks both simultaneously — the economic wound and the human cost.

In [4]:
# Animated scatter: mfg job loss vs deaths of despair change
anim_scatter = panel[
    (panel["year"] >= 2000) &
    panel["mfg_change_pct_2000"].notna() &
    panel["ddr_change_since_2000"].notna()
].copy()

# Add population for bubble size (use 2016 pop as proxy if available)
pop_2016 = state_panel[state_panel["year"] == 2016][["state_abbr","population"]].rename(columns={"population": "pop"})
anim_scatter = anim_scatter.merge(pop_2016, on="state_abbr", how="left")
anim_scatter["pop"] = anim_scatter["pop"].fillna(3_000_000)

fig = px.scatter(
    anim_scatter,
    x="mfg_change_pct_2000",
    y="ddr_change_since_2000",
    animation_frame="year",
    text="state_abbr",
    color="region",
    size="pop",
    size_max=45,
    color_discrete_map={
        "Rust Belt / Appalachia": "#e74c3c",
        "Rust Belt": "#c0392b",
        "Appalachia": "#e67e22",
        "Coastal Metro": "#2980b9",
        "Other": "#7f8c8d",
    },
    range_x=[-55, 20],
    range_y=[-5, 45],
    title="Manufacturing Job Loss vs. Rising Deaths of Despair Since 2000",
    labels={
        "mfg_change_pct_2000": "% Change in Manufacturing Jobs (since 2000)",
        "ddr_change_since_2000": "Increase in Deaths of Despair per 100k (since 2000)",
        "region": "Region",
    },
)
fig.update_traces(textposition="top center")
fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=1)
fig.add_vline(x=0, line_dash="dash", line_color="gray", line_width=1)
fig.add_annotation(x=-45, y=42, text="⬅ Jobs Gone, People Dying", showarrow=False,
    font=dict(size=12, color="red"))
fig.update_layout(height=580, hovermode="closest")
fig.show()


## Chapter 4: The Geography of Abandonment

Not every state suffered equally. The crisis carved its deepest wounds into the communities that built this country with their hands — and were left behind when the economy moved on.

In [5]:
# Regional comparison — violin plot with individual state points
latest_year = int(state_panel.year.max())
latest = state_panel[state_panel["year"] == latest_year].copy()

fig = px.strip(
    latest.dropna(subset=["deaths_despair_rate","region"]),
    x="region", y="deaths_despair_rate",
    color="region",
    hover_name="state_abbr",
    hover_data={"overdose_death_rate": ":.1f", "suicide_rate": ":.1f", "region": False},
    color_discrete_map={
        "Rust Belt / Appalachia": "#e74c3c",
        "Rust Belt": "#c0392b",
        "Appalachia": "#e67e22",
        "Coastal Metro": "#2980b9",
        "Other": "#7f8c8d",
    },
    title=f"Deaths of Despair by Region — {latest_year} Snapshot",
    labels={"deaths_despair_rate": "Deaths per 100k", "region": ""},
)

# Add regional medians
for region in latest["region"].dropna().unique():
    med = latest[latest["region"]==region]["deaths_despair_rate"].median()
    fig.add_hline(y=med, line_dash="dot", line_color="black", line_width=1,
                  annotation_text=f"{region} median: {med:.1f}", annotation_position="right")

fig.update_layout(height=500, showlegend=False)
fig.show()


## Chapter 5: Two Americas

Below is the starkest visualization: comparing states that kept their manufacturing base versus those that lost it. The gap between these two groups has grown every single year since 1999.

In [6]:
# Split states: high mfg loss vs low mfg loss
base_2000_all = state_panel[state_panel["year"] == 2000][["state_abbr","mfg_index_2000","manufacturing_employees_thousands"]]
state_2016 = state_panel[state_panel["year"] == state_panel["year"].max()][["state_abbr","manufacturing_employees_thousands"]]

# Compute total mfg loss from 2000 to latest
mfg_change = base_2000[["state_abbr","mfg_2000"]].merge(
    state_panel[state_panel["year"]==latest_year][["state_abbr","manufacturing_employees_thousands"]],
    on="state_abbr"
)
mfg_change["loss_pct"] = (mfg_change["manufacturing_employees_thousands"] - mfg_change["mfg_2000"]) / mfg_change["mfg_2000"] * 100
top_losers = set(mfg_change.nsmallest(20, "loss_pct")["state_abbr"])
top_keepers = set(mfg_change.nlargest(20, "loss_pct")["state_abbr"])

state_panel["mfg_group"] = state_panel["state_abbr"].apply(
    lambda x: "High Mfg Loss (top 20)" if x in top_losers
    else ("Low Mfg Loss (bottom 20)" if x in top_keepers else "Middle")
)

two_americas = (
    state_panel[state_panel["mfg_group"].isin(["High Mfg Loss (top 20)", "Low Mfg Loss (bottom 20)"])]
    .groupby(["year","mfg_group"])["deaths_despair_rate"]
    .mean()
    .reset_index()
)

fig = px.line(
    two_americas,
    x="year", y="deaths_despair_rate",
    color="mfg_group",
    markers=True,
    color_discrete_map={
        "High Mfg Loss (top 20)": "#e74c3c",
        "Low Mfg Loss (bottom 20)": "#2980b9",
    },
    title="Two Americas: States That Lost vs. Kept Manufacturing Jobs",
    labels={"deaths_despair_rate": "Avg Deaths of Despair per 100k", "year": "Year", "mfg_group": ""},
)
fig.update_layout(height=450, legend=dict(orientation="h", y=1.05))
# Shade the gap
fig.add_vrect(x0=2013, x1=2016, fillcolor="rgba(231,76,60,0.1)", line_width=0,
    annotation_text="Gap widens<br>with fentanyl", annotation_position="top right")
fig.show()


## Chapter 6: The Toll

Numbers can be abstract. Here's what this data represents in human lives.

In [7]:
# The human cost
nat_clean = national_panel.dropna(subset=["overdose_deaths_total","suicide_deaths_total"])
total_od = int(nat_clean["overdose_deaths_total"].sum())
total_su = int(nat_clean["suicide_deaths_total"].sum())
total_combined = total_od + total_su

worst_state = state_panel[state_panel["year"]==latest_year].nlargest(1, "deaths_despair_rate").iloc[0]

print("=" * 60)
print("THE TOLL — 1999 to 2016")
print("=" * 60)
print(f"Drug overdose deaths recorded:    {total_od:>12,}")
print(f"Suicide deaths recorded:          {total_su:>12,}")
print(f"Combined deaths of despair:       {total_combined:>12,}")
print()
print(f"That's more than 10× the American lives lost in Vietnam.")
print(f"Every year since 2000, the rate has risen relentlessly.")
print()
print(f"Hardest hit state in {latest_year}:")
print(f"  {worst_state['state_name']}: {worst_state['deaths_despair_rate']:.1f} per 100,000 people")
print(f"  (overdose: {worst_state['overdose_death_rate']:.1f} | suicide: {worst_state['suicide_rate']:.1f})")

# Bar chart: annual total deaths
fig = go.Figure()
fig.add_trace(go.Bar(
    x=nat_clean["year"], y=nat_clean["overdose_deaths_total"],
    name="Drug Overdose Deaths", marker_color="#e74c3c",
))
fig.add_trace(go.Bar(
    x=nat_clean["year"], y=nat_clean["suicide_deaths_total"],
    name="Suicide Deaths", marker_color="#8e44ad",
))
fig.update_layout(
    barmode="stack",
    title="Annual Deaths of Despair — Drug Overdoses + Suicides (1999–2016)",
    xaxis_title="Year",
    yaxis_title="Number of Deaths",
    legend=dict(orientation="h", y=1.05),
    height=430,
    plot_bgcolor="#fafafa",
)
fig.show()


THE TOLL — 1999 to 2016
Drug overdose deaths recorded:         632,331
Suicide deaths recorded:               649,843
Combined deaths of despair:          1,282,174

That's more than 10× the American lives lost in Vietnam.
Every year since 2000, the rate has risen relentlessly.

Hardest hit state in 2016:
  West Virginia: 71.3 per 100,000 people
  (overdose: 52.0 | suicide: 19.3)


## Methodology & Sources

**What are 'Deaths of Despair'?**
The term was coined by economists Anne Case and Angus Deaton to describe deaths from drug overdoses, suicide, and chronic liver disease/alcohol abuse — a trifecta that has disproportionately hit middle-aged, non-college-educated, white Americans in deindustrializing regions.

This study covers drug overdoses and suicides specifically (the two largest components with the cleanest state-level data).

**Data Sources:**
- **CDC NCHS Drug Poisoning Mortality by State** (1999–2016): Age-adjusted death rates per 100,000. Dataset IDs: `jx6g-fdh6`, `xbxb-epbu` (CDC data.cdc.gov).
- **CDC NCHS Leading Causes of Death** (1999–2020): Suicide rates per 100,000. Dataset ID: `bi63-dtpu` (CDC data.cdc.gov).
- **FRED/BLS**: State manufacturing employment (monthly, annual average computed). Series format: `{ABBR}MFG`.
- **Census ACS 1-year**: Median household income and poverty rate by state, 2005–2023.

**Limitations:**
- Data covers 1999–2016 (CDC NCHS age-adjusted rates). The fentanyl acceleration post-2016 is not fully captured.
- Manufacturing employment includes all manufacturing sectors; some states have more capital-intensive vs. labor-intensive manufacturing.
- Correlation does not imply causation. Other factors (prescribing patterns, social capital, mental health infrastructure) also contribute.
- Age-adjusted rates allow comparison across states with different age distributions.

**Methodology:**
- Deaths of despair rate = overdose death rate + suicide rate (both age-adjusted per 100k)
- Manufacturing job loss = % change from year 2000 baseline
- Regional classification: Rust Belt (OH, PA, MI, IN, WI, IL, MO, NY); Appalachia overlap states coded as 'Rust Belt / Appalachia'; coastal metros (CA, NY, MA, WA, OR, CT, NJ, MD) coded separately